In [ ]:
import os
import json
import pickle
import random

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

import cv2

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

## Configuration

In [ ]:
DATASET_DIR = "abstrait-v4"
FICHIER_CNN = "cnn_features_pca.npy"
FICHIER_PKL_COLLEGUE = "features.pkl"
FICHIER_BONUS = "bonus_features_brutes.npy"

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

POIDS_CNN = 0.7
POIDS_CLASSIQUE = 0.3
POIDS_BONUS = 0.05

K_CHOISI = 15
SEUIL_ARTISTES_MAJEURS = 20
MIN_OEUVRES = 3

ARTISTES_JURY = ['Accardi', 'Riley', 'Vasarely', 'Hantai', 'Toroni', 'Klee']

## 1. Extraction des features bonus

In [ ]:
image_names_cnn = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith(IMAGE_EXTENSIONS)]

bonus_features = []

for nom in image_names_cnn:
    path = os.path.join(DATASET_DIR, nom)

    taille_octets = os.path.getsize(path)
    img_gray = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        raise ValueError(f"Erreur de lecture pour l'image {nom}")

    h, w = img_gray.shape
    ratio_compression = taille_octets / (h * w)

    grille_4x4 = cv2.resize(img_gray, (4, 4), interpolation=cv2.INTER_AREA).astype(float)
    symetrie = (np.mean(np.abs(grille_4x4[:, 0] - grille_4x4[:, 3]))
                + np.mean(np.abs(grille_4x4[:, 1] - grille_4x4[:, 2])))

    img_small = cv2.resize(img_gray, (256, 256))
    f = np.fft.fft2(img_small)
    fshift = np.fft.fftshift(f)
    magnitude = 20 * np.log(np.abs(fshift) + 1)
    magnitude[128-10:128+10, 128-10:128+10] = 0
    seuil_pics = np.mean(magnitude) + 3 * np.std(magnitude)
    nb_pics_fft = np.sum(magnitude > seuil_pics)

    bonus_features.append([ratio_compression, symetrie, nb_pics_fft])

X_bonus_brut = np.array(bonus_features)
np.save(FICHIER_BONUS, X_bonus_brut)
print(f"Features bonus extraites. Dimensions : {X_bonus_brut.shape}")

## 2. Chargement et fusion multimodale

In [ ]:
X_cnn = np.load(FICHIER_CNN) if os.path.exists(FICHIER_CNN) else (_ for _ in ()).throw(FileNotFoundError(f"Fichier {FICHIER_CNN} introuvable."))

with open(FICHIER_PKL_COLLEGUE, "rb") as f:
    data_collegue = pickle.load(f)

if isinstance(data_collegue, tuple):
    if isinstance(data_collegue[0], (list, np.ndarray)) and isinstance(data_collegue[0][0], str):
        noms_collegue, X_collegue_brut = data_collegue[0], np.array(data_collegue[1])
    else:
        X_collegue_brut, noms_collegue = np.array(data_collegue[0]), data_collegue[1]
else:
    df_collegue = pd.DataFrame(data_collegue)
    noms_collegue = df_collegue.index.tolist() if isinstance(df_collegue.index[0], str) else df_collegue.iloc[:, 0].tolist()
    X_collegue_brut = df_collegue.select_dtypes(include=[np.number]).values

dict_features_collegue = {nom: vecteur for nom, vecteur in zip(noms_collegue, X_collegue_brut)}

X_collegue_aligne = []
for nom in image_names_cnn:
    cles_possibles = [k for k in dict_features_collegue.keys() if nom in k]
    if cles_possibles:
        X_collegue_aligne.append(dict_features_collegue[cles_possibles[0]])
    else:
        raise ValueError(f"Image {nom} introuvable dans {FICHIER_PKL_COLLEGUE} !")
X_collegue_aligne = np.array(X_collegue_aligne)

n_comp = X_cnn.shape[1]
X_collegue_clean = np.nan_to_num(X_collegue_aligne, nan=np.nanmedian(X_collegue_aligne))
X_collegue_reduit = PCA(n_components=n_comp).fit_transform(StandardScaler().fit_transform(X_collegue_clean))

X_bonus_brut = np.load(FICHIER_BONUS) if os.path.exists(FICHIER_BONUS) else (_ for _ in ()).throw(FileNotFoundError(f"Fichier {FICHIER_BONUS} introuvable."))
X_bonus_scaled = StandardScaler().fit_transform(X_bonus_brut)

X_final = np.hstack((
    X_collegue_reduit * POIDS_CLASSIQUE,
    X_cnn * POIDS_CNN,
    X_bonus_scaled * POIDS_BONUS
))
print(f"Matrice multimodale prête : {X_final.shape}")

## 3. Clustering K-Means

In [ ]:
print(f"Lancement du clustering sur l'espace multimodal (K={K_CHOISI})...")
kmeans_fusion = KMeans(n_clusters=K_CHOISI, random_state=42, n_init=10)
cluster_labels = kmeans_fusion.fit_predict(X_final)

score_silhouette = silhouette_score(X_final, cluster_labels)
print(f"Score de Silhouette (Fusion) : {score_silhouette:.4f}")

artistes = [nom.split('_')[0].capitalize() for nom in image_names_cnn]
df_results = pd.DataFrame({'Artiste': artistes, 'Cluster': cluster_labels})

print("Répartition globale des œuvres par cluster :")
print(df_results['Cluster'].value_counts().sort_index())

## 4. Heatmap artistes × clusters

In [ ]:
artistes = [nom.split('_')[0].capitalize() for nom in image_names_cnn]
df_results = pd.DataFrame({'Artiste': artistes, 'Cluster': cluster_labels})

comptage_artistes = df_results['Artiste'].value_counts()
artistes_majeurs = comptage_artistes[comptage_artistes >= SEUIL_ARTISTES_MAJEURS].index
df_majeurs = df_results[df_results['Artiste'].isin(artistes_majeurs)]

crosstab = pd.crosstab(df_majeurs['Artiste'], df_majeurs['Cluster'])
crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100

plt.figure(figsize=(14, 10))
sns.heatmap(crosstab_pct, annot=True, fmt=".0f", cmap="Blues",
            cbar_kws={'label': "% des œuvres de l'artiste"})
plt.title("Répartition Stylistique (Fusion CNN 0.85 / Classique 0.15 / Bonus 0.40)",
          fontsize=16, fontweight='bold', pad=20)
plt.xlabel("ID du Cluster (K-Means)", fontsize=12)
plt.ylabel("Artistes Majeurs", fontsize=12)
plt.tight_layout()
plt.show()

## 5. Grille de similarité (plus proches voisins)

In [ ]:
NUM_TARGETS = 8
N_NEIGHBORS = 6
N_COLS = N_NEIGHBORS + 1

nn_model = NearestNeighbors(n_neighbors=N_COLS, metric='cosine')
nn_model.fit(X_final)

random_indices = random.sample(range(len(image_names_cnn)), NUM_TARGETS)

fig, axes = plt.subplots(NUM_TARGETS, N_COLS, figsize=(20, 3 * NUM_TARGETS))
fig.suptitle(f"Similarité multimodale : {NUM_TARGETS} Cibles et leurs {N_NEIGHBORS} Plus Proches Voisins",
             fontsize=18, fontweight='bold', y=0.98)

for row, idx in enumerate(random_indices):
    target_vector = X_final[idx].reshape(1, -1)
    distances, indices = nn_model.kneighbors(target_vector)

    for col in range(N_COLS):
        ax = axes[row, col]
        voisin_idx = indices[0][col]
        voisin_name = image_names_cnn[voisin_idx]
        voisin_artiste = voisin_name.split('_')[0].capitalize()
        img_path = os.path.join(DATASET_DIR, voisin_name)

        try:
            img = mpimg.imread(img_path)
            ax.imshow(img)
            if col == 0:
                ax.set_title(f"CIBLE\n{voisin_artiste}", color='darkblue', fontweight='bold', fontsize=12)
                for spine in ax.spines.values():
                    spine.set_color('darkblue')
                    spine.set_linewidth(4)
            else:
                ax.set_title(f"{voisin_artiste}\n(d={distances[0][col]:.2f})", fontsize=10)
                for spine in ax.spines.values():
                    spine.set_color('gray')
                    spine.set_linewidth(1)
        except FileNotFoundError:
            ax.set_title("Image absente", fontsize=9)

        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.subplots_adjust(top=0.92, hspace=0.4)
plt.show()

## 6. Matrice de distances inter-artistes (jury)

In [ ]:
noms_extraits = np.array([nom.split('_')[0].capitalize() for nom in image_names_cnn])

centroides = {}
for artiste in ARTISTES_JURY:
    indices = np.where(noms_extraits == artiste)[0]
    if len(indices) > 0:
        centroides[artiste] = np.mean(X_final[indices], axis=0)
        print(f"✔ {artiste} : {len(indices)} œuvres trouvées.")
    else:
        print(f"✗ {artiste} : absent de la base.")

artistes_trouves = list(centroides.keys())
matrice_centroides = np.array(list(centroides.values()))
distances = cosine_distances(matrice_centroides)

plt.figure(figsize=(10, 8))
sns.heatmap(distances, xticklabels=artistes_trouves, yticklabels=artistes_trouves,
            annot=True, fmt=".3f", cmap="YlOrRd",
            cbar_kws={'label': 'Distance Cosinus (0 = Identique)'})
plt.title("Matrice des Distances Inter-Artistes\n(Espace Multimodal)", fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Rapport JSON final

In [ ]:
kmeans_final = KMeans(n_clusters=K_CHOISI, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_final)
score_silhouette = silhouette_score(X_final, cluster_labels)

artistes = [nom.split('_')[0].capitalize() for nom in image_names_cnn]
df_results = pd.DataFrame({'Artiste': artistes, 'Cluster': cluster_labels})

cluster_balance = {f"Cluster_{k}": int(v) for k, v in df_results['Cluster'].value_counts().sort_index().items()}

noms_extraits = np.array(artistes)
centroides = {a: np.mean(X_final[np.where(noms_extraits == a)[0]], axis=0)
              for a in ARTISTES_JURY if len(np.where(noms_extraits == a)[0]) > 0}

distances_dict = {}
if len(centroides) > 1:
    noms_trouves = list(centroides.keys())
    dist_matrix = cosine_distances(np.array(list(centroides.values())))
    distances_dict = {
        art_A: {art_B: round(float(dist_matrix[i, j]), 4) for j, art_B in enumerate(noms_trouves)}
        for i, art_A in enumerate(noms_trouves)
    }

artistes_a_garder = set(df_results['Artiste'].value_counts()[lambda x: x >= SEUIL_ARTISTES_MAJEURS].index.tolist() + ARTISTES_JURY)
df_majeurs = df_results[df_results['Artiste'].isin(artistes_a_garder)]
crosstab_artistes = pd.crosstab(df_majeurs['Artiste'], df_majeurs['Cluster'])
repartition_propre = {
    artiste: {str(k): int(v) for k, v in distribution.items()}
    for artiste, distribution in crosstab_artistes.to_dict(orient='index').items()
}

rapport_final = {
    "1_Configuration": {
        "Dimensions_Espace": int(X_final.shape[1]),
        "Nombre_de_Clusters_K": int(K_CHOISI),
        "Score_de_Silhouette": round(float(score_silhouette), 4)
    },
    "2_Equilibre_des_Clusters": cluster_balance,
    "3_Distances_Jury": distances_dict,
    "4_Repartition_Artistes_Cles": repartition_propre
}

print("=" * 80)
print(json.dumps(rapport_final, indent=4))
print("=" * 80)

## 8. Parangons dynamiques (images les plus centrales par cluster)

In [ ]:
kmeans_final = KMeans(n_clusters=K_CHOISI, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_final)
distances_to_centers = kmeans_final.transform(X_final)

clusters_a_afficher = list(range(K_CHOISI))

fig, axes = plt.subplots(len(clusters_a_afficher), 3, figsize=(15, 4 * len(clusters_a_afficher)))
fig.suptitle("Parangons Dynamiques : Centres Géométriques du Modèle Hybridé",
             fontsize=18, fontweight='bold', y=1.02)

for i, cluster_id in enumerate(clusters_a_afficher):
    indices_dans_cluster = np.where(cluster_labels == cluster_id)[0]

    if len(indices_dans_cluster) > 0:
        ordre_tri = np.argsort(distances_to_centers[indices_dans_cluster, cluster_id])
        top_3 = indices_dans_cluster[ordre_tri[:3]]

        print(f"Cluster {cluster_id} ({len(indices_dans_cluster)} œuvres) :")
        for rank, idx in enumerate(top_3):
            nom_image = image_names_cnn[idx]
            artiste = nom_image.split('_')[0].capitalize()
            dist_val = distances_to_centers[idx, cluster_id]
            print(f"  {rank+1}. {nom_image} | Distance : {dist_val:.4f}")

            ax = axes[i, rank]
            chemin_image = os.path.join(DATASET_DIR, nom_image)
            if os.path.exists(chemin_image):
                ax.imshow(mpimg.imread(chemin_image))
                ax.set_title(f"Proto {rank+1} : {artiste}", fontsize=11, pad=8)
            else:
                ax.text(0.5, 0.5, f"Image introuvable\n{nom_image}", ha='center', va='center', fontsize=9)
            ax.axis('off')

    axes[i, 0].text(-0.15, 0.5, f"Cluster {cluster_id}",
                    rotation=90, va='center', ha='right',
                    transform=axes[i, 0].transAxes, fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Analyse de concentration des artistes par cluster

In [ ]:
total_par_artiste = df_results['Artiste'].value_counts()
artistes_valides = total_par_artiste[total_par_artiste >= MIN_OEUVRES].index
df_filtre = df_results[df_results['Artiste'].isin(artistes_valides)]

repartition_brute = pd.crosstab(df_filtre['Artiste'], df_filtre['Cluster'])
repartition_pct = repartition_brute.div(repartition_brute.sum(axis=1), axis=0) * 100

seuils = [
    ("100% (Exclusivité totale)", 100.0),
    (">= 90% (Très forte concentration)", 90.0),
    (">= 75% (Forte concentration)", 75.0),
    ("> 50% (Majorité simple)", 50.0001),
]

resultats_par_cluster = {k: {nom: [] for nom, _ in seuils} for k in range(K_CHOISI)}

for artiste in repartition_pct.index:
    for cluster_id in repartition_pct.columns:
        pct = repartition_pct.loc[artiste, cluster_id]
        if pct > 50:
            total_oeuvres = repartition_brute.loc[artiste, cluster_id]
            total_artiste = total_par_artiste[artiste]
            if pct == 100:   cat = seuils[0][0]
            elif pct >= 90:  cat = seuils[1][0]
            elif pct >= 75:  cat = seuils[2][0]
            else:            cat = seuils[3][0]
            resultats_par_cluster[cluster_id][cat].append(
                f"{artiste} ({pct:.1f}% -> {total_oeuvres}/{total_artiste} toiles)"
            )

print("ANALYSE DE CONCENTRATION DES ARTISTES PAR SEUILS\n" + "=" * 80)
for cluster_id in range(K_CHOISI):
    if any(len(resultats_par_cluster[cluster_id][nom]) > 0 for nom, _ in seuils):
        print(f"\nCLUSTER {cluster_id}")
        for nom_seuil, _ in seuils:
            artistes_liste = resultats_par_cluster[cluster_id][nom_seuil]
            if artistes_liste:
                print(f"   {nom_seuil} :")
                for art in sorted(artistes_liste, reverse=True):
                    print(f"      -> {art}")
print("\n" + "=" * 80)

## 10. Anatomie des clusters (feature la plus discriminante)

In [ ]:
noms_features = (
    [f"PCA_Classique_{i+1}" for i in range(50)]
    + [f"PCA_CNN_{i+1}" for i in range(50)]
    + ["Ratio_Compression_JPEG", "Symetrie_Grille_4x4", "Pics_FFT_Motifs"]
)

moyenne_globale = np.mean(X_final, axis=0)
std_globale = np.std(X_final, axis=0) + 1e-9
centroides = kmeans_final.cluster_centers_

print("ANATOMIE DES CLUSTERS : FEATURE LA PLUS DISCRIMINANTE\n" + "=" * 70)
for k in range(K_CHOISI):
    z_scores = (centroides[k] - moyenne_globale) / std_globale
    idx_max = np.argmax(np.abs(z_scores))
    sens = "Supérieure" if z_scores[idx_max] > 0 else "Inférieure"
    print(f"Cluster {k:2d} | Feature : {noms_features[idx_max]:<25} | Tendance : {sens:<10} (Z-Score: {z_scores[idx_max]:+.2f})")
print("=" * 70)